# LLM Run Coach: Data Analysis & Performance Feedback
Use Snowflake Cortex LLM to analyze running data, detect abnormalities, and provide coaching advice.

*Co-authored with CoCo*

In [1]:
# from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
import subprocess
import shutil
import json
import textwrap

try:
    from fpdf import FPDF
except ImportError:
    FPDF = None

# session = get_active_session()

print('Session ready - using statistical analysis for coaching insights')

Session ready - using statistical analysis for coaching insights


In [ ]:
# Prepare data summary for LLM analysis
csv_path = Path("STAGE_RUNPROJECT.csv")
df = pd.read_csv(csv_path)
df['DATE'] = pd.to_datetime(df['DATE'])

# Convert category values if needed
if df['CONDITION_CATEGORY'].dtype != object:
    df['CONDITION_CATEGORY'] = df['CONDITION_CATEGORY'].astype(int)
    df['CONDITION_CATEGORY'] = df['CONDITION_CATEGORY'].map({0: 'Regular Training', 1: 'Event/Race'})

# Compute per-session aggregates
session_stats = df.groupby(['DATE', 'CONDITION_CATEGORY']).agg(
    total_laps=('LAPS', 'max'),
    avg_pace=('AVG_PACE_MIN_KM', 'mean'),
    max_pace=('AVG_PACE_MIN_KM', 'max'),
    min_pace=('AVG_PACE_MIN_KM', 'min'),
    avg_hr=('AVG_HR_BPM', 'mean'),
    max_hr=('AVG_HR_BPM', 'max'),
    avg_cadence=('AVG_RUN_CADENCE_SPM', 'mean'),
    avg_stride=('AVG_STRIDE_LENGTH_M', 'mean'),
    total_ascent=('TOTAL_ASCENT_M', 'max'),
    total_time=('CUMULATIVE_TIME', 'max')
).reset_index()

lap_stats = df.groupby(['CONDITION_CATEGORY', 'LAPS']).agg(
    mean_pace=('AVG_PACE_MIN_KM', 'mean'),
    std_pace=('AVG_PACE_MIN_KM', 'std'),
    mean_hr=('AVG_HR_BPM', 'mean'),
    mean_cadence=('AVG_RUN_CADENCE_SPM', 'mean'),
    mean_stride=('AVG_STRIDE_LENGTH_M', 'mean')
).reset_index()

# Category distribution counts
cat_counts = df.groupby('CONDITION_CATEGORY')['DATE'].nunique().to_dict()

# Detect outliers (pace > 2 std from mean per lap group)
merged = df.merge(
    df.groupby(['CONDITION_CATEGORY', 'LAPS'])['AVG_PACE_MIN_KM'].agg(['mean', 'std']).reset_index(),
    on=['CONDITION_CATEGORY', 'LAPS']
)
merged['z_score'] = (merged['AVG_PACE_MIN_KM'] - merged['mean']) / merged['std'].replace(0, 1)
outliers = merged[merged['z_score'].abs() > 2][['DATE', 'CONDITION_CATEGORY', 'LAPS', 'AVG_PACE_MIN_KM', 'z_score']].copy()

summary_stats = {
    'date_range': f"{df['DATE'].min().date()} to {df['DATE'].max().date()}",
    'total_sessions': int(df['DATE'].nunique()),
    'total_lap_records': int(len(df)),
    'missing_values': int(df.isnull().sum().sum()),
    'category_distribution': {
        'Regular Training': int(cat_counts.get('Regular Training', 0)),
        'Event/Race': int(cat_counts.get('Event/Race', 0)),
    },
    'outlier_count': int(len(outliers)),
}
context_lines = [f'Sessions: {len(session_stats)}',
f'Outlier laps detected: {summary_stats["outlier_count"]}',
f'Date range: {summary_stats["date_range"]}']
for i in session_stats:
    print(i)
data_context="\n".join(context_lines)

Sessions: 12
Outlier laps detected: 6
Date range: 2026-04-30 to 2026-06-21


In [ ]:
# Build context string for Model and report generation
context_lines = [
    f"## Running Data Summary",
    f"Date range: {summary_stats['date_range']}",
    f"Total sessions: {summary_stats['total_sessions']}",
    f"Total lap records: {summary_stats['total_lap_records']}",
    f"Missing values: {summary_stats['missing_values']}",
    "",
    "## Per-Session Overview (most recent 5 sessions):",
    session_stats.sort_values('DATE', ascending=False).head(5).to_string(index=False),
    "",
    "## Average Metrics by Lap Number:",
    lap_stats.to_string(index=False),
    "",
    "## Statistical Outliers (pace > 2 std deviations from lap mean):",
    outliers.to_string(index=False) if len(outliers) > 0 else 'No significant outliers detected.',
    "",
    "## Pace Trend (session averages over time):",
    session_stats[['DATE', 'CONDITION_CATEGORY', 'avg_pace', 'avg_hr', 'avg_cadence']].sort_values('DATE').to_string(index=False),
]

temdata_context = "\n".join(context_lines)
data_context=data_context+"\n\n" + temdata_context
print(f'Context length: {len(data_context)} chars')
print(data_context[:500] + '...')

Context length: 4100 chars
## Running Data Summary
Date range: 2026-04-30 to 2026-06-21
Total sessions: 12
Total lap records: 120
Missing values: 0

## Per-Session Overview (most recent 5 sessions):
      DATE CONDITION_CATEGORY  total_laps  avg_pace  max_pace  min_pace  avg_hr  max_hr  avg_cadence  avg_stride  total_ascent  total_time
2026-06-21         Event/Race        10.0  7.141667  9.116667  5.633333   167.1   175.0        154.2       0.925           9.0       85.70
2026-06-18   Regular Training        10.0  9.00833...


In [ ]:
# 1) DATA ANALYSIS: Quality assessment and abnormality detection
print('=' * 70)
print('SECTION 1: DATA ANALYSIS & ABNORMALITIES')
print('=' * 70)

# Dataset overview
print(f'\n--- Dataset Quality ---')
print(f'Date range: {summary_stats["date_range"]}')
print(f'Total sessions: {summary_stats["total_sessions"]}')
print(f'Total lap records: {summary_stats["total_lap_records"]}')
print(f'Laps per session: {df.groupby("DATE")["LAPS"].count().mean():.0f} (consistent)')
print(f'Missing values: {summary_stats["missing_values"]} total across all columns')

# Category distribution
print(f'\n--- Category Distribution ---')
for cat_label, count in summary_stats['category_distribution'].items():
    print(f'{cat_label}: {count} sessions')

# Statistical patterns
print(f'\n--- Key Statistical Patterns ---')
for cat_label in ['Regular Training', 'Event/Race']:
    cat_data = df[df['CONDITION_CATEGORY'] == cat_label]
    if len(cat_data) == 0:
        continue
    print(f'\n{cat_label}:')
    print(f'  Avg pace: {cat_data["AVG_PACE_MIN_KM"].mean():.2f} min/km (std: {cat_data["AVG_PACE_MIN_KM"].std():.2f})')
    print(f'  Avg HR: {cat_data["AVG_HR_BPM"].mean():.0f} bpm (std: {cat_data["AVG_HR_BPM"].std():.0f})')
    print(f'  Avg cadence: {cat_data["AVG_RUN_CADENCE_SPM"].mean():.0f} spm')
    print(f'  Avg stride: {cat_data["AVG_STRIDE_LENGTH_M"].mean():.2f} m')

# Outlier detection (z-score > 2)
print(f'\n--- Abnormalities & Outliers ---')
for cat_label in ['Regular Training', 'Event/Race']:
    cat_data = df[df['CONDITION_CATEGORY'] == cat_label].reset_index(drop=True)
    for lap in sorted(cat_data['LAPS'].unique()):
        lap_subset = cat_data[cat_data['LAPS'] == lap].reset_index(drop=True)
        pace_vals = lap_subset['AVG_PACE_MIN_KM']
        if pace_vals.std() > 0 and len(pace_vals) >= 3:
            z_scores = np.abs((pace_vals - pace_vals.mean()) / pace_vals.std())
            for idx in z_scores[z_scores > 2].index:
                d = lap_subset.loc[idx, 'DATE'].date()
                v = lap_subset.loc[idx, 'AVG_PACE_MIN_KM']
                print(f'  [{cat_label}] Lap {int(lap)}, {d}: pace {v:.2f} min/km (z>2 from mean {pace_vals.mean():.2f})')

# Trend analysis
print(f'\n--- Trends ---')
for cat_label in ['Regular Training', 'Event/Race']:
    cat_data = df[df['CONDITION_CATEGORY'] == cat_label]
    session_paces = cat_data.groupby('DATE')['AVG_PACE_MIN_KM'].mean().sort_index()
    if len(session_paces) >= 3:
        x = np.arange(len(session_paces))
        slope, _, r_value, p_value, _ = stats.linregress(x, session_paces.values)
        direction = 'IMPROVING (pace decreasing)' if slope < 0 else 'DECLINING (pace increasing)'
        print(f'  {cat_label}: {direction} | slope={slope:.3f} min/km per session | R²={r_value**2:.2f} | p={p_value:.3f}')

# PDF report helper

def build_pdf_report(output_path: Path, report_data: dict, df: pd.DataFrame, session_stats: pd.DataFrame, lap_stats: pd.DataFrame, outliers: pd.DataFrame):
    if FPDF is None:
        raise ImportError('fpdf package is required for PDF report generation. Install with pip install fpdf')

    pdf = FPDF()
    pdf.add_page()
    pdf.set_font('Arial', 'B', 16)
    
    pdf.cell(0, 10, 'Run Coach Data Analysis Report', ln=True)
    pdf.set_font('Arial', '', 11)
    pdf.cell(0, 8, f"Date range: {report_data['date_range']}", ln=True)
    pdf.cell(0, 8, f"Total sessions: {report_data['total_sessions']}", ln=True)
    pdf.cell(0, 8, f"Total lap records: {report_data['total_lap_records']}", ln=True)
    pdf.cell(0, 8, f"Missing values: {report_data['missing_values']}", ln=True)
    pdf.ln(4)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 8, 'Dataset Overview', ln=True)
    pdf.set_font('Arial', '', 11)
    pdf.multi_cell(0, 6, f"Sessions: {report_data['total_sessions']}, range {report_data['date_range']}, laps: {report_data['total_lap_records']}")
    pdf.multi_cell(0, 6, f"Missing values: {report_data['missing_values']}")
    pdf.ln(2)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 8, 'Quality Assessment', ln=True)
    pdf.set_font('Arial', '', 11)
    pdf.multi_cell(0, 6, 'This report includes data quality assessment, category coverage, and abnormality detection.')
    pdf.multi_cell(0, 6, f"Missing values found: {report_data['missing_values']}")
    pdf.ln(2)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 8, 'Category Distribution', ln=True)
    pdf.set_font('Arial', '', 11)
    for cat_label, count in report_data['category_distribution'].items():
        pdf.multi_cell(0, 6, f"{cat_label}: {count} sessions")
    pdf.ln(2)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 8, 'Statistical Patterns', ln=True)
    pdf.set_font('Arial', '', 11)
    for cat_label in ['Regular Training', 'Event/Race']:
        cat_data = df[df['CONDITION_CATEGORY'] == cat_label]
        if len(cat_data) == 0:
            continue
        pdf.multi_cell(0, 6, f"{cat_label}: Avg pace {cat_data['AVG_PACE_MIN_KM'].mean():.2f} ± {cat_data['AVG_PACE_MIN_KM'].std():.2f}, Avg HR {cat_data['AVG_HR_BPM'].mean():.0f} ± {cat_data['AVG_HR_BPM'].std():.0f}, Cadence {cat_data['AVG_RUN_CADENCE_SPM'].mean():.0f}, Stride {cat_data['AVG_STRIDE_LENGTH_M'].mean():.2f}")
    pdf.ln(2)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 8, 'Abnormality Detection', ln=True)
    pdf.set_font('Arial', '', 11)
    pdf.multi_cell(0, 6, f"Outliers detected: {report_data['outlier_count']}")
    if len(outliers) > 0:
        for _, row in outliers.head(10).iterrows():
            pdf.multi_cell(0, 6, f"  - {row['DATE'].date()} | {row['CONDITION_CATEGORY']} | Lap {int(row['LAPS'])} | Pace {row['AVG_PACE_MIN_KM']:.2f} | z={row['z_score']:.2f}")
        if len(outliers) > 10:
            pdf.multi_cell(0, 6, f"... plus {len(outliers) - 10} more outliers")
    else:
        pdf.multi_cell(0, 6, 'No significant pace outliers detected.')
    pdf.ln(2)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 8, 'Trend Analysis', ln=True)
    pdf.set_font('Arial', '', 11)
    for cat_label in ['Regular Training', 'Event/Race']:
        cat_data = df[df['CONDITION_CATEGORY'] == cat_label]
        session_paces = cat_data.groupby('DATE')['AVG_PACE_MIN_KM'].mean().sort_index()
        if len(session_paces) >= 3:
            x = np.arange(len(session_paces))
            slope, _, r_value, p_value, _ = stats.linregress(x, session_paces.values)
            direction = 'improving' if slope < 0 else 'declining'
            pdf.multi_cell(0, 6, f"{cat_label}: {direction} slope={slope:.3f}, R²={r_value**2:.2f}, p={p_value:.3f}")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    pdf.output(str(output_path))
    print(f'PDF report generated at: {output_path}')

# Generate the PDF report
report_file = Path('run_coach_data_analysis_report.pdf')
build_pdf_report(report_file, summary_stats, df, session_stats, lap_stats, outliers)


SECTION 1: DATA ANALYSIS & ABNORMALITIES

--- Dataset Quality ---
Date range: 2026-04-30 to 2026-06-21
Total sessions: 12
Total lap records: 120
Laps per session: 10 (consistent)
Missing values: 0 total across all columns

--- Category Distribution ---
Regular Training: 10 sessions
Event/Race: 2 sessions

--- Key Statistical Patterns ---

Regular Training:
  Avg pace: 9.67 min/km (std: 2.90)
  Avg HR: 137 bpm (std: 17)
  Avg cadence: 139 spm
  Avg stride: 0.79 m

Event/Race:
  Avg pace: 7.29 min/km (std: 0.95)
  Avg HR: 168 bpm (std: 8)
  Avg cadence: 156 spm
  Avg stride: 0.89 m

--- Abnormalities & Outliers ---
  [Regular Training] Lap 2, 2026-05-17: pace 9.83 min/km (z>2 from mean 7.88)
  [Regular Training] Lap 3, 2026-06-14: pace 11.80 min/km (z>2 from mean 8.25)
  [Regular Training] Lap 6, 2026-06-12: pace 18.88 min/km (z>2 from mean 10.96)
  [Regular Training] Lap 7, 2026-06-12: pace 15.38 min/km (z>2 from mean 9.36)
  [Regular Training] Lap 8, 2026-05-17: pace 11.02 min/km (z>2 

In [ ]:
# 2) RUN COACH: Ollama-backed performance analysis and interactive answer generation
print('=' * 70)
print('SECTION 2: RUN COACH - INTERACTIVE OLLAMA ANALYSIS')
print('=' * 70)

import shutil
import subprocess
from pathlib import Path
import textwrap
import pandas as pd  # Fixed: Missing import


def run_shell_command(command, input_text=None, capture_output=True):
    return subprocess.run(
        command, 
        input=input_text, 
        text=True, 
        encoding='utf-8',           # Force UTF-8 decoding
        errors='replace',           # Prevent crashes if weird bytes appear
        capture_output=capture_output
    )


def ollama_available():
    return shutil.which('ollama') is not None


def model_exists(model_name='run-coach-model'):
    if not ollama_available():
        return False
    result = run_shell_command(['ollama', 'list'])
    return model_name in result.stdout


def find_resource_files():
    csv_files = sorted(Path('.').glob('*.csv'))
    pdf_files = sorted(Path('.').glob('*.pdf'))
    return csv_files, pdf_files


def load_csv_summaries(csv_files):
    summaries = []
    for csv_path in csv_files:
        try:
            sample = pd.read_csv(csv_path, nrows=5)
            with open(csv_path, encoding='utf-8', errors='ignore') as f:
                row_count = sum(1 for _ in f) - 1
            summaries.append({
                'file': csv_path.name,
                'rows': int(row_count),
                'columns': list(sample.columns),
                'sample': sample.to_csv(index=False)
            })
        except Exception as exc:
            summaries.append({'file': csv_path.name, 'error': str(exc)})
    return summaries


def load_pdf_texts(pdf_files):
    # Fixed: Try modern 'pypdf' first, fallback to 'PyPDF2'
    reader_cls = None
    try:
        import pypdf
        reader_cls = pypdf.PdfReader
    except ImportError:
        try:
            import PyPDF2
            reader_cls = PyPDF2.PdfReader
        except ImportError:
            print('Neither pypdf nor PyPDF2 is installed; PDF contents cannot be read. Install with pip install pypdf')
            return []

    texts = []
    for pdf_path in pdf_files:
        try:
            reader = reader_cls(str(pdf_path))
            pages = []
            for page in reader.pages:
                page_text = page.extract_text() or ''
                if page_text:
                    pages.append(page_text)
            texts.append({
                'file': pdf_path.name,
                'text': '\n'.join(pages)[:8000]
            })
        except Exception as exc:
            texts.append({'file': pdf_path.name, 'error': str(exc)})
    return texts


def create_run_coach_modelfile(modelfile: Path):
    # Fixed: Replaced invalid YAML format with valid Ollama Modelfile syntax
    modelfile.write_text(textwrap.dedent('''
        FROM llama3:latest

        SYSTEM """
        You are an expert running coach.
        Use the available CSV and PDF resources in the current directory when answering.
        If a question can be answered from a PDF, cite the PDF file name as the reference.
        If the PDF does not contain the answer, use the CSV running data and the run coach model.
        Answers should be concise, practical, and evidence-based.
        """
    ''').strip())


def create_run_coach_model(model_name='run-coach-model'):
    modelfile = Path('RunCoachModelfile')
    create_run_coach_modelfile(modelfile)
    print(f"Creating Ollama model '{model_name}' with modelfile {modelfile}")
    result = run_shell_command(['ollama', 'create', model_name, '-f', str(modelfile)])
    if result.returncode != 0:
        raise RuntimeError(f'Ollama model creation failed:\n{result.stderr}')
    print(result.stdout)


def build_resource_context(data_context: str = ""):
    csv_files, pdf_files = find_resource_files()
    csv_summaries = load_csv_summaries(csv_files)
    pdf_texts = load_pdf_texts(pdf_files)

    lines = ['## Available CSV Resources']
    if csv_summaries:
        for summary in csv_summaries:
            if 'error' in summary:
                lines.append(f"- {summary['file']}: ERROR reading file: {summary['error']}")
            else:
                lines.append(f"- {summary['file']}: {summary['rows']} rows, columns={summary['columns']}")
                lines.append('  Sample rows:')
                lines.append(summary['sample'])
    else:
        lines.append('- No CSV files found.')

    lines.append('')
    lines.append('## Available PDF Resources')
    if pdf_texts:
        for pdf in pdf_texts:
            if 'error' in pdf:
                lines.append(f"- {pdf['file']}: ERROR reading PDF: {pdf['error']}")
            else:
                lines.append(f"- {pdf['file']}: extracted text sample below")
                lines.append(pdf['text'])
    else:
        lines.append('- No PDF files found or PDF parser unavailable.')

    return (data_context + '\n\n' + '\n'.join(lines)).strip()


def ask_run_coach_question(question, data_context="", model_name='run-coach-model'):
    # Fixed: Added default data_context="" to prevent NameError
    resource_context = build_resource_context(data_context)
    prompt = textwrap.dedent(f"""
        Use the provided dataset summary and available file resources to answer the question.
        If a PDF contains the answer, explicitly cite the PDF file name in the response.
        If the information is not present in any PDF, answer from CSV running data and the run coach model.

        Resource context:
        {resource_context}

        Question:
        {question}
    """)
    result = run_shell_command(['ollama', 'run', model_name], input_text=prompt)
    if result.returncode != 0:
        raise RuntimeError(f'Ollama run failed:\n{result.stderr}')
    return result.stdout.strip()


model_name = 'run-coach-model'
if ollama_available():
    if not model_exists(model_name):
        create_run_coach_model(model_name)
    print('Local Ollama model is ready. Generating interactive analysis now...')
    interactive_summary = ask_run_coach_question(
        'Provide a run coach summary, identify abnormalities and outliers, and suggest actionable improvements using the current dataset.'
    )
    print('\n' + interactive_summary)
    with open("output.txt", "a", encoding="utf-8") as file:
        file.write(interactive_summary)
else:
    print('Ollama CLI not found on PATH. Install Ollama and retry to enable model-backed interactive answers.')
    print('This notebook will still generate the PDF report and analysis summary.')
    print('Ollama CLI not found on PATH. Install Ollama and retry to enable model-backed interactive answers.')
    print('This notebook will still generate the PDF report and analysis summary.')


SECTION 2: RUN COACH - INTERACTIVE OLLAMA ANALYSIS
Local Ollama model is ready. Generating interactive analysis now...

**Run Coach Summary**

Based on the provided CSV files (PRED_CHART_DATA.csv and STAGE_RUNPROJECT.c
STAGE_RUNPROJECT.csv) and the PDF report (run_coach_data_analysis_report.pd
(run_coach_data_analysis_report.pdf), here's a summary of the data:

* The dataset consists of 12 sessions, with a total of 120 lap records.
* Regular Training is the most common session type, accounting for 10 out o
of 12 sessions.
* Average pace and heart rate values are generally consistent across sessio
sessions, with some minor variations.

**Abnormalities and Outliers**

The PDF report highlights several abnormalities and outliers:

1. **Session-specific outliers**: Six outliers were detected across three R
Regular Training sessions (2026-05-17, 2026-06-12, and 2026-04-30). These o
outliers are characterized by unusually high or low pace values.
2. **Lap-specific outliers**: No specific lap

AttributeError: 'str' object has no attribute 'to_csv'

In [ ]:
# 3) RUN COACH: Interactive follow-up prompts using the saved Ollama model
print('=' * 70)
print('SECTION 3: RUN COACH - INTERACTIVE Q&A')
print('=' * 70)

if ollama_available():
    follow_up_questions = [
        'What are the most important data quality issues I should fix first?',
        'Which laps show the largest deviations and why?',
        'Recommend three workouts that match the current performance data and explain why.',
    ]
    for q in follow_up_questions:
        print(f'\nQUESTION: {q}')
        print('-' * 60)
        try:
            answer = ask_run_coach_question(q)
            print(answer)
        except RuntimeError as exc:
            print(f'Error running Ollama question: {exc}')
            break
else:
    print('Interactive Ollama question answering is unavailable because the Ollama CLI is not installed or not on PATH.')
    print('Install Ollama and rerun the notebook to enable model-backed interactive responses.')